# Step 6 — Gaze direction: a model vs. a naive rule

`step_03` gave us iris offset (`gaze_x`, `gaze_y`) and head features
(`nose_dx`, `nose_dy`, `roll_deg`). The simplest way to turn `gaze_x`/
`gaze_y` into a direction is a threshold rule, same style as `step_02`'s
blink rule. This notebook checks that rule two ways:

1. Against a **random forest** that also sees head features, on real
   hand-labelled gaze-direction segments.
2. Against a **free ground-truth clip** (`head_only_<name>.mp4`) where the truth
   isn't a guess: the eyes stay fixed on one point while the head turns,
   so by the experiment's own design every frame's true gaze is "centre"
   — nobody needs to label this one by hand.

**Known limitation, stated up front rather than discovered the hard way:**
vertical gaze (up/down) is much weaker and noisier in this kind of footage
than horizontal (left/right). Expect the up/down numbers below to look
worse — that contrast *is* a real result, not something gone wrong.


## At a glance

| | |
|---|---|
| **Workshop step** | 6 of 6 |
| **Needs in this folder** | for each `eyes_only` clip: `features_eyes_only_<name>.csv` (from step 3) and its label file `labels_eyes_only_<name>__<labeller>.csv` (from step 4, gaze mode). Also `features_head_only_<name>.csv` (from step 3) for the control check. In a group, one set per person. |
| **Produces** | nothing saved to disk. Prints a report for each method, a confusion matrix, and the `head_only` check. |
| **Run time** | a few seconds |
| **Next** | `STUDENT_TASKS.md` (pick one open-ended task) |

**How to run:** in the menu choose *Run > Restart Kernel and Run All Cells*, or press Shift+Enter on each cell from the top down. Do not skip cells: later cells use variables made by earlier ones.

If the first code cell stops with "No labelled eyes_only clips found", you have not done step 4 in gaze mode yet. That message is expected, not a bug.

**Working in a group:** this notebook uses every labelled `eyes_only` clip it finds, so the model can be trained on more than one person. See `TEST_CLIP` in the first code cell. The labels must use **left and right as seen in the video** (the convention from `README.md` and step 4). The naive rule below is written for that convention, so labels made the other way round would turn the comparison upside down without any error message.

## Load the hand labels and the features

In [ ]:
import glob
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# ---------------------------------------------------------------------
# TRY IT (groups): which clip is held back for testing?
#   None                every look (segment) from every clip goes into one
#                       pool, and 30% of the looks are held back.
#   "eyes_only_<name>"  that whole clip is held back. The model trains on
#                       everyone else's clips and is tested on a person
#                       it has never seen. Needs at least two labelled clips.
# Run the notebook once per person in your group as the held-out clip.
# Does a model trained on your eyes work on someone else's?
# ---------------------------------------------------------------------
TEST_CLIP = None

# Only needed if one clip was labelled by several people: put the labeller's
# name here to choose whose labels to use. None = use the only file present.
LABELLER = None

# TRY IT: a frame counts as a blink (and is dropped) when its EAR is below
# this fraction of the clip's median EAR. Try 0.4 and 0.8. Too low keeps
# blink frames in, too high throws away real "looking down" frames.
BLINK_EAR_FRACTION = 0.6


def find_label_file(clip_name):
    """Return the label file for one clip, or None if it is not labelled yet."""
    matches = sorted(glob.glob(f"labels_{clip_name}__*.csv"))
    if len(matches) > 1 and LABELLER:
        matches = [m for m in matches if m == f"labels_{clip_name}__{LABELLER}.csv"]
    if len(matches) > 1:
        raise SystemExit(f"{clip_name} has {len(matches)} label files: {matches}. Set LABELLER to pick one.")
    return matches[0] if matches else None


labelled = {}   # clip name -> (features per frame, labelled intervals)
for feat_path in sorted(glob.glob("features_eyes_only_*.csv")):
    clip_name = os.path.basename(feat_path)[len("features_"):-len(".csv")]
    labels_path = find_label_file(clip_name)
    if labels_path is None:
        print(f"Skipping {clip_name}: no label file yet (step 4, gaze mode)")
        continue
    labelled[clip_name] = (pd.read_csv(feat_path), pd.read_csv(labels_path))
    print(f"{clip_name}: {len(labelled[clip_name][1])} labelled intervals over "
          f"{len(labelled[clip_name][0])} frames (labels from {labels_path})")

if not labelled:
    raise SystemExit("No labelled eyes_only clips found. Run step 3, then step_04_label_intervals.py with MODE = 'gaze'.")

### One label per frame, grouped into segments

A *segment* is one unbroken stretch with the same label (one "look"). Blink frames, frames without a face and frames nobody labelled are dropped. With several clips, all their segments go into one table.

In [ ]:
# ---------------------------------------------------------------------
# TRY IT: FEATURE_COLS is everything the model is allowed to look at.
#   - Remove the three head columns (nose_dx, nose_dy, roll_deg). Does the
#     head_only check at the end of the notebook get worse?
#   - Swap gaze_x / gaze_y for the per-eye columns r_x, l_x, r_y, l_y.
#     Do two separate eyes tell the model more than their average?
# ---------------------------------------------------------------------
FEATURE_COLS = ["gaze_x", "gaze_y", "nose_dx", "nose_dy", "roll_deg"]
CLASSES = ["centre", "left", "right", "up", "down"]


def stamp_frame_labels(intervals, n_frames):
    """Turn labelled intervals into one label per frame (None = not labelled)."""
    labels = np.full(n_frames, None, dtype=object)
    for _, row in intervals.iterrows():
        labels[int(row.start_frame):int(row.end_frame) + 1] = row.label
    return labels


def drop_unusable_frames(df):
    """Drop blink frames and frames without a face.

    Iris position is meaningless while the eyes are closed, so those
    frames would only teach the model noise.
    """
    df = df.dropna(subset=FEATURE_COLS + ["ear"])
    return df[df["ear"] >= BLINK_EAR_FRACTION * df["ear"].median()].copy()


parts = []
for clip_name, (feats, intervals) in labelled.items():
    feats["label"] = stamp_frame_labels(intervals, len(feats))
    feats["clip"] = clip_name
    # a new segment starts whenever the label changes. The clip name is part
    # of the id, so segments from different clips can never be mixed up.
    segment_number = (feats["label"] != feats["label"].shift()).cumsum()
    feats["segment_id"] = clip_name + ":" + segment_number.astype(str)
    feats = feats[feats["label"].notna()]
    parts.append(drop_unusable_frames(feats))

feats = pd.concat(parts, ignore_index=True)
print(f"{len(feats)} usable frames in {feats.segment_id.nunique()} segments from {feats['clip'].nunique()} clip(s)")
pd.crosstab(feats["clip"], feats["label"])

## The naive rule being challenged

Same style as `step_02`'s blink threshold: fixed cutoffs on `gaze_x`/
`gaze_y`. Iris offsets are small (very roughly ±0.3 eye widths at the
extremes), so the cutoffs are small numbers too. The starting values below
are rough guesses and are **not** tuned to your face or camera. This is
still a real baseline, not a strawman: it's a perfectly reasonable first
guess for turning iris offset into a direction.

"Left" and "right" mean **left and right as seen in the video**, the same
convention as the labels from step 4. A positive `gaze_x` means the iris
sits towards the right of the picture, so the rule calls it "right".

The rule uses one set of cutoffs for everybody. With several people in the
data, ask yourself whether that can work.

In [ ]:
# ---------------------------------------------------------------------
# TRY IT: these three cutoffs ARE the naive rule (units: eye widths).
#   GX_THRESH  gaze_x beyond +/- this value is called "right" / "left"
#              (as seen in the video: positive gaze_x = right of the picture)
#   GY_UP      gaze_y below this value is called "up"   (negative = up)
#   GY_DOWN    gaze_y above this value is called "down"
# Open the plot from step_03: where do YOUR gaze_x and gaze_y sit when
# you look left, right, up and down, and where do they rest? Set the
# cutoffs from that and see how much the rule improves. Does a setting
# that suits one person in your group suit the others?
# ---------------------------------------------------------------------
GX_THRESH, GY_UP, GY_DOWN = 0.12, -0.07, 0.06


def naive_rule(gaze_x, gaze_y):
    out = np.full(len(gaze_x), "centre", dtype=object)
    out[gaze_y > GY_DOWN] = "down"
    out[gaze_y < GY_UP] = "up"
    # left / right = as seen in the video, matching the step 4 labels.
    # They are checked last, so they win over up/down.
    out[gaze_x > GX_THRESH] = "right"   # iris towards the right of the picture
    out[gaze_x < -GX_THRESH] = "left"   # iris towards the left of the picture
    return out


# TRY IT: TEST_SIZE is the share of segments held back when TEST_CLIP is
# None. SPLIT_SEED decides WHICH segments those are. Change the seed to 1,
# 2, 3 and rerun: how much do the scores below move? That movement is a
# rough measure of how far you can trust any single number here.
TEST_SIZE = 0.3
SPLIT_SEED = 0

if TEST_CLIP is None:
    # split by whole segment, not by frame -- same reasoning as step_05's
    # split in time: neighbouring frames of one "look" are near-identical
    seg_ids = feats["segment_id"].unique()
    if len(seg_ids) < 6:
        raise SystemExit(f"Only {len(seg_ids)} labelled segments: too few to hold some back for testing. "
                         "Label every separate look as its own interval in step 4 (about 3 per direction), or record a longer clip.")
    train_ids, test_ids = train_test_split(seg_ids, test_size=TEST_SIZE, random_state=SPLIT_SEED)
    train = feats[feats.segment_id.isin(train_ids)]
    test = feats[feats.segment_id.isin(test_ids)]
    print(f"Split by segment: {TEST_SIZE:.0%} of all segments held back for testing")
else:
    if TEST_CLIP not in labelled:
        raise SystemExit(f"TEST_CLIP = {TEST_CLIP!r} is not a labelled clip. Choose from {list(labelled)}.")
    if len(labelled) < 2:
        raise SystemExit("Holding out a whole clip needs at least two labelled clips. Set TEST_CLIP = None.")
    train = feats[feats["clip"] != TEST_CLIP]
    test = feats[feats["clip"] == TEST_CLIP]
    print(f"Held-out clip: testing on {TEST_CLIP}, training on the other {train['clip'].nunique()} clip(s)")

print(f"Train: {len(train)} frames ({train.segment_id.nunique()} segments)")
print(f"Test:  {len(test)} frames ({test.segment_id.nunique()} segments)")

## Method 1: the naive threshold rule

In [ ]:
base_pred = naive_rule(test["gaze_x"].values, test["gaze_y"].values)
print("--- Naive threshold rule ---")
print(classification_report(test["label"], base_pred, labels=CLASSES, zero_division=0))


## Method 2: random forest

In [ ]:
# TRY IT: a random forest is many decision trees voting.
#   n_estimators  how many trees. Try 10 and 500.
#   max_depth     how many yes/no questions each tree may ask. Try 2 (very
#                 simple trees) and None (no limit). A deep forest can
#                 memorise the training looks: does the test score follow?
clf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=0, class_weight="balanced")
clf.fit(train[FEATURE_COLS], train["label"])
model_pred = clf.predict(test[FEATURE_COLS])
print("--- Random forest (gaze + head features) ---")
print(classification_report(test["label"], model_pred, labels=CLASSES, zero_division=0))

# which features did the forest lean on most? (the shares add up to 1)
for col, importance in sorted(zip(FEATURE_COLS, clf.feature_importances_), key=lambda pair: -pair[1]):
    print(f"  importance {col:<10} {importance:.2f}")

cm = confusion_matrix(test["label"], model_pred, labels=CLASSES)
pd.DataFrame(cm, index=CLASSES, columns=CLASSES)

## Reading the confusion matrix

Rows are the true label, columns are what the model predicted. A perfect
model would have every count on the diagonal. Off-diagonal entries tell
you *what gets confused with what* — e.g. a big number in the
"true=up, predicted=centre" cell says the model often can't tell a slight
upward look from looking straight ahead.

## The free robustness check

The `head_only` clips were not labelled by hand. By how they were recorded
(eyes fixed on one point, head turning), the true gaze label for *every*
frame is "centre". Any frame either method calls something else is a
head-pose confusion, not a real gaze read. Each `head_only` clip in the
folder is checked separately.

In [ ]:
control_paths = sorted(glob.glob("features_head_only_*.csv"))
if not control_paths:
    print("No features_head_only_*.csv found: record a head_only clip and rerun step 3 to get this check.")

print("True label on every head_only clip: 100% centre")
for path in control_paths:
    clip_name = os.path.basename(path)[len("features_"):-len(".csv")]
    control = drop_unusable_frames(pd.read_csv(path))
    base_control = naive_rule(control["gaze_x"].values, control["gaze_y"].values)
    model_control = clf.predict(control[FEATURE_COLS])
    print(f"{clip_name} ({len(control)} frames):")
    print(f"  naive rule: {(base_control == 'centre').mean():.1%} correctly called centre")
    print(f"  model:      {(model_control == 'centre').mean():.1%} correctly called centre")

## Discuss

- Did giving the model head features (`nose_dx`, `nose_dy`, `roll_deg`)
  actually help it resist being fooled by head turning on the `head_only`
  clips, or did it make little difference? What would you check next
  either way?
- Look at the up/down rows of the confusion matrix specifically. Does the
  weakness match the warning at the top of this notebook?
- In a group: compare the pooled split (`TEST_CLIP = None`) with holding
  out each person in turn. A model can look good on people it has seen
  and fail on a new one. Which number is the honest one for an assistive
  device that a new child would use?
- This whole comparison rests on **a few short clips and one labeller
  each**. How much would you trust a "model wins" or "model loses"
  conclusion here? (Same question as `step_05` — see `STUDENT_TASKS.md`
  tasks 1a/1b.)

Look back through the code cells for the `TRY IT` comments: each one marks
a value that is yours to change.

If the vertical-gaze weakness bothers you, `STUDENT_TASKS.md` task 3
(fixing the resting `gaze_y` offset) is a direct, concrete next step.